# RecoverKv on LongGenBench-16K: long-form generation (branch `Clustered_int2_paused_fast`)

**What this measures.** LongGenBench (Wu et al., ICLR'25), 16K setting. Each document starts from a ~700-token prompt
that asks for one long structured text: 52 weekly diary entries, 52 weekly menus, a 100-floor skyscraper, or a
100-block city. It runs to at most 16,000 new tokens. The prompt also sets Specific Task Instructions that must show up
in particular entries:
* once ("Floor 51 is a graphic design studio");
* over a range ("weeks 24–25 at a fashion week in Paris");
* periodically ("a parenting workshop every 5 weeks from week 13").

The prompt is tiny, so almost everything the model attends to is its own output: this is a test of the decode-phase
cache. At your settings the fp16 tier holds about 1,090 keys, so the int2 tier and the read gate start about 400
generated tokens in, and keys are first dropped once the sequence passes about 9,600 tokens. That is arithmetic from
`resolve()`; step 10 prints the exact figures for the real prompts.

Two arms see byte-identical prompts:
* **fullkv**: HF `DynamicCache`, flash-attention-2.
* **ours**: the windowed two-tier cache at your settings (step 8), flash backend, read gate on.
* *(optional)* **ctrl_gate1.0**: ours at `quant_gate_ratio = 1.0`, the gate's control arm.

**The protocol** is the reference code's (`mozhu621/LongGenBench` @ `45c320e`):
* The dataset prompt goes in verbatim: no chat template, one BOS.
* Generation stops at `*** finished` (cut from the text, as vLLM does), at EOS, or at 16,000 new tokens.
* The scored text is `prefix + generation`, split into entries on `#*#`.
* A judge (Llama-3.1-8B-Instruct, as the paper says) answers one yes/no question per instruction whose entry exists,
  using `eval.py`'s few-shot prompt.

**Two choices where the paper and the code disagree.**
* **Decoding.** The paper says greedy; `inference.py` samples at T = 0.95. Greedy is the default (`DECODING`), because
  only a deterministic decode makes `ours − fullkv` a statement about the cache rather than about two random draws.
* **Judge decoding.** The paper does not state it; `eval.py` samples. Greedy is the default (`JUDGE_DECODING`), with one
  question per forward pass, so a verdict never depends on a draw or on what else shared a batch.

**Metrics** (the paper's Table 2, from `eval.py` / `static.ipynb`):
* **CR**: completion rate, the share of the required entries that exist.
* **STIC-1**: yes ÷ instructions judged.
* **STIC-2**: yes ÷ all instructions.
* **wAvg**: CR × STIC-2, the paper's headline.
* once / range / periodic accuracies, and Len (word count).

**Why a number here can be trusted.** Each claim is checked, not assumed:

| claim | how it is checked |
|---|---|
| the data is the benchmark | SHA-256 of the file at HF revision `3c96cac` (byte-identical to the GitHub copy); 6,205 instructions at 16K, as the authors' own table says |
| the prompt and stop rule are the reference's | the prompt is passed through untouched; stop string, EOS ids, `prefix + generation` and one BOS are recorded in every sidecar and re-checked |
| the metric is the reference's | CPU tests pin block parsing (first `"<type> N"`, first block wins), the judge prompt character for character, the `'yes' in response` rule and every aggregate |
| the judge works | a 4-question known-answer selftest every time the judge loads; any miss stops that part |
| **ours ran the gated method** | `read_gate.verdict == "gated"` with `gated == fired > 0` in every part, plus the realised read fraction |
| ours is at your operating point | every cache knob is re-read from the sidecar and compared with step 8, including the resolved geometry |
| the cache was really compressed | per document: whether the int2 tier was used, whether keys were dropped, and the KV-byte fraction held |
| each verdict belongs to its document | the judge file records the SHA of the generations it judged; a regenerated part invalidates its old verdicts |

Run the cells in order, from step 1. Each **part** (one type × one chunk of documents × one arm) is generated, judged,
verified and copied to Google Drive as it finishes. If you run the notebook again, parts that are already complete and
verified (same settings, commit and GPU model) are reused instead of re-run.

**Before step 1:**
* *Runtime → Change runtime type → A100 GPU.*
* Accept the licence at https://huggingface.co/meta-llama/Llama-3.1-8B-Instruct, then add your HF token as a Colab
  secret named `HF_TOKEN` (key icon in the left bar) and allow this notebook to read it.

### How long it takes (one A100, B = 1)
These are arithmetic estimates, not measurements. They assume about 13,000 new tokens per document (Llama-3.1-8B averages
~8,800 words at 16K in the paper, and Floor and Block documents usually run to the 16,000 cap). They also assume about
30 ms/token for fullkv and 55 ms/token for ours. After each part, step 11 re-projects the rest from what it measured.

| `N_PER_TYPE` | documents | fullkv | ours | judge (both arms) | total |
|---|---|---|---|---|---|
| **5** (default) | 20 | ~2.2 h | ~4 h | ~15 min | **~6.5 h** |
| 10 | 40 | ~4.3 h | ~8 h | ~30 min | ~13 h |
| 100 (the whole benchmark) | 400 | ~43 h | ~80 h | ~5 h | ~5 days |

The paper runs all 400 documents with vLLM on eight GPUs. Here every document is decoded at batch size 1 through the
cache under test, so a Colab run is a subset: the first `N_PER_TYPE` documents of each type, the same ones in both arms.
Raising `N_PER_TYPE` in multiples of `CHUNK` reuses every part already done. A Colab disconnect costs at most the part
that was running (about 30 to 60 min).

## 1. GPU check

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv
!python --version

## 2. Clone the branch, pinned to the commit that adds this runner
Your efficiency notebook pins `7beb3da`. This one pins `704aff5`, the first commit with this LongGenBench runner.
Between the two, two commits touched the cache code:
* `db9a7ed` added the completion-ablation knob, off by default (`quant_completion: null` is the shipped kernel path),
  and an output-dtype option in the decode kernel that only an ablation arm sets.
* `5924f03` moved the int2 quantiser's trace gate from torch 2.7 to 2.8. That changes nothing on the torch 2.6 this
  notebook installs (step 6 asserts 2.6.x).

The other commits in between only add the LongGenBench files.

Every "ours" part is checked for the default: `quant_completion.mode == "native"` and `read_gate.completion_applied == 0`.
Strictly, though, the speed table and this accuracy table then come from adjacent commits. Say so if you quote them
side by side, or re-run the efficiency notebook at `704aff5`.

In [ ]:
%cd /content
!rm -rf RecoverKv
!git clone -q --branch Clustered_int2_paused_fast --single-branch https://github.com/shaswata965-coder/RecoverKv.git
%cd /content/RecoverKv
!git checkout -q 704aff5d75cdfe9942649c04a4c79735e2734c9c && git log -1 --format='%h %s'
!test -f modules/evaluation/longgenbench_judge.py && echo "LongGenBench runner and judge present"

## 3. Install the pinned stack (~5 min)
torch 2.6 is the build the recorded GPU runs used; transformers must be exactly 4.47.1
(`utils/cache_factory.py` refuses anything newer). flash-attn comes from its prebuilt wheel. If pip
starts *compiling* flash-attn, stop the cell, because that means no wheel matched this Python.

**When this cell finishes: Runtime → Restart session**, then continue at step 4.

In [ ]:
%%bash
set -euo pipefail
pip -q install torch==2.6.0 torchvision==0.21.0 torchaudio==2.6.0 --index-url https://download.pytorch.org/whl/cu124

# flash-attn: prebuilt wheel matching this python / torch 2.6 / C++ ABI
read PYV ABI < <(python -c "import sys,torch; print(f'cp{sys.version_info[0]}{sys.version_info[1]}', 'TRUE' if torch._C._GLIBCXX_USE_CXX11_ABI else 'FALSE')")
ok=0
for V in 2.7.4.post1 2.8.3; do
  WHL="https://github.com/Dao-AILab/flash-attention/releases/download/v${V}/flash_attn-${V}+cu12torch2.6cxx11abi${ABI}-${PYV}-${PYV}-linux_x86_64.whl"
  echo "trying $WHL"
  if pip -q install --no-deps "$WHL"; then ok=1; break; fi
done
[ "$ok" = 1 ] || { echo "no prebuilt flash-attn wheel for ${PYV}/torch2.6/abi${ABI}"; exit 1; }

pip -q install "transformers==4.47.1" "tokenizers>=0.21,<0.22" "huggingface_hub>=0.24,<1.0" \
               "datasets>=2.16" "accelerate>=0.26" sentencepiece protobuf pyyaml einops pytest
python -c "import torch, transformers, flash_attn; print('torch', torch.__version__, '| transformers', transformers.__version__, '| flash_attn', flash_attn.__version__)"
echo "NOW: Runtime -> Restart session, then run step 4."

## 4. Hugging Face login + model download (~16 GB)

In [ ]:
from google.colab import userdata
from huggingface_hub import login, snapshot_download
login(token=userdata.get("HF_TOKEN"))
snapshot_download("meta-llama/Llama-3.1-8B-Instruct",
                  allow_patterns=["*.json", "*.safetensors", "tokenizer*"])  # skips the 16 GB original/*.pth

## 5. LongGenBench data, pinned
`Dataset_short.json` (16K) and `Dataset_long.json` (32K) from `mozhu/LongGenBench` at revision `3c96cac`, checked
against a SHA-256 each (`data/longgenbench_loader.py`). About 3 MB.

In [ ]:
%%bash
set -euo pipefail
cd /content/RecoverKv
python -m data.longgenbench_loader --out data/longgenbench

## 6. Preflight
Every check here guards something the run would otherwise do silently wrong, or refuse hours in:
* the transformers pin, and the softmax LSE that L-reuse needs from flash-attn;
* Inductor compiling on this box;
* the read gate having a production call site, and the shipped configs being at the operating point;
* the tokenizer adding exactly one BOS to a raw prompt;
* the stop string being supported by this `generate`;
* the pinned data being the benchmark (400 documents, 6,205 instructions at 16K).

Then the CPU unit tests that pin the protocol, the judge prompt and every metric to the reference code.
It must print `PREFLIGHT OK`.

In [ ]:
%%bash
set -euo pipefail
cd /content/RecoverKv
python - <<'EOF'
import re, subprocess, sys, torch, transformers, triton, flash_attn
print("python", sys.version.split()[0], "| torch", torch.__version__, "cuda", torch.version.cuda,
      "| triton", triton.__version__, "| transformers", transformers.__version__,
      "| flash_attn", flash_attn.__version__)
assert torch.__version__.startswith("2.6"), "torch must be 2.6.x"
from utils.cache_factory import assert_transformers_version_supported
assert_transformers_version_supported()                     # == 4.47.1
p = torch.cuda.get_device_properties(0)
print(f"gpu {p.name}  {p.total_memory / 2**30:.1f} GiB")
from flash_attn import flash_attn_func                       # L-reuse needs the LSE back
q = torch.randn(1, 128, 32, 128, device="cuda", dtype=torch.float16)
k = torch.randn(1, 128, 8, 128, device="cuda", dtype=torch.float16)
out = flash_attn_func(q, k, k, causal=True, return_attn_probs=True)
assert isinstance(out, tuple) and tuple(out[1].shape) == (1, 32, 128), "flash-attn returned no LSE"
f = torch.compile(lambda x: (x * 2).sin().sum(-1))             # the eviction is compiled-or-raise
f(torch.randn(64, 64, device="cuda")); torch.cuda.synchronize()

# The read gate has a production call site (CLAUDE.md: "It is wired. Check, don't assume.")
grep = subprocess.run("grep -rn 'fused_gate' --include='*.py' . | grep -v '^./tests/'", shell=True,
                      capture_output=True, text=True).stdout.splitlines()
calls = [l for l in grep if re.search(r"=\s*fused_gate\(", l)]
assert calls, "fused_gate has no non-test call site: the decode step would not be gated"
print("read gate call site:", ":".join(calls[0].split(":")[:2]))
r = subprocess.run([sys.executable, "scripts/check_operating_point.py"], capture_output=True, text=True)
assert r.returncode == 0, r.stdout[-3000:]
print("shipped configs at the operating point")

# One BOS on a raw prompt; this generate supports stop strings; the model's EOS ids.
from transformers import AutoTokenizer, GenerationConfig
from utils.prompting import encode_prompt
import data.longgenbench_loader as L
tok = AutoTokenizer.from_pretrained("meta-llama/Llama-3.1-8B-Instruct")
doc = L.load_type("data/longgenbench", "16K", "Week", 0, 1)[0]
ids = encode_prompt(tok, doc["prompt"], return_tensors="pt").input_ids[0].tolist()
assert ids[0] == tok.bos_token_id and ids[1] != tok.bos_token_id, ids[:4]
assert hasattr(GenerationConfig(), "stop_strings"), "this transformers has no stop_strings"
eos = GenerationConfig.from_pretrained("meta-llama/Llama-3.1-8B-Instruct").eos_token_id
assert sorted(eos) == [128001, 128008, 128009], eos
for s in L.SETTINGS:
    rows = L.load_all("data/longgenbench", s)
    n = sum(len(r["checks_once"]) + len(r["checks_range"]) + len(r["checks_periodic"]) for r in rows)
    print(f"  {s}: {len(rows)} documents, {n} instructions, sha {L.dataset_sha('data/longgenbench', s)[:12]}")
EOF
LONGGENBENCH_DATA=data/longgenbench python -m pytest -q -p no:cacheprovider tests/test_longgenbench.py
echo "PREFLIGHT OK"

## 7. Mount Google Drive
Every part is copied here as soon as it finishes, and earlier verified parts are reused from here,
so nothing is lost if the runtime is recycled.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

## 8. Settings (the only cell you edit)
The cache knobs are your efficiency notebook's settings, which are the shipped operating point
(`utils.config.OPERATING_POINT`). The budget is always `CACHE_BUDGET × (prompt + max_new_tokens)`. That is the method's
own definition, the same as `BUDGET_BASIS = "prefill_plus_gen"` there; there is no `"context"` basis in a quality run.
At 16K that is about 3,300 tokens' worth of bytes. A document that stops before 16,000 tokens holds more than
`CACHE_BUDGET` of its own sequence, and the table reports by how much.

* `N_PER_TYPE`: the first N documents of each type, the same in every arm. Keep it a multiple of `CHUNK`.
* `CHUNK`: documents per part. Keep it fixed between reruns, or earlier parts no longer match and are not reused.
* `DECODING`, `JUDGE_DECODING`: `"greedy"` (default) or `"repo_sampling"` (the reference code's T = 0.95, top_p = 0.95).

In [ ]:
SETTING      = "16K"           # "16K" (up to 16,000 new tokens) | "32K" (32,000)
TYPES        = None            # None = all four: Week, Floor, Menu Week, Block (16K)
N_PER_TYPE   = 5               # first N documents of each type; 100 = the whole benchmark
CHUNK        = 5               # documents per part
DECODING     = "greedy"        # "greedy" (the paper) | "repo_sampling" (inference.py)
WINDOW_SIZE  = 8               # multiple of 4 when QUANT_RATIO > 0
NUM_SINK     = 5
LOCAL_WINDOW = 128             # int multiple of WINDOW_SIZE
QUANT_RATIO  = 0.70            # int2 share of the byte budget; 0 = fp16 only (no gate)
GATE_RATIO   = 0.25            # fraction of int2 windows read per step, (0, 1]
CACHE_BUDGET = 0.20            # fraction of (prompt + max_new_tokens) kept, in bytes
QUANT_MODE   = "bytes"
DTYPE        = "float16"
MODEL_ID     = "meta-llama/Llama-3.1-8B-Instruct"
JUDGE_MODEL  = "meta-llama/Llama-3.1-8B-Instruct"   # the paper's judge
JUDGE_DECODING = "greedy"      # "greedy" | "repo_sampling" (eval.py)
CONTROL_ROW  = False           # True adds ours at quant_gate_ratio 1.0 (the gate's control arm)
REUSE_VERIFIED = True          # reuse earlier parts with identical settings/commit/GPU that verify
READ_FRACTION_TOL = 0.05       # warn if the realised read fraction is further than this from GATE_RATIO

## 9. Helpers (run as-is)

In [ ]:
import json, os, shutil, subprocess, sys, time, importlib, hashlib
from pathlib import Path
import numpy as np

REPO = "/content/RecoverKv"
PIN = "704aff5d75cdfe9942649c04a4c79735e2734c9c"
sys.path.insert(0, REPO)
import data.longgenbench_loader as L
import modules.evaluation.longgenbench_scoring as S
L, S = importlib.reload(L), importlib.reload(S)

PY = shutil.which("python") or sys.executable
DATA_DIR = f"{REPO}/data/longgenbench"
ENV = dict(os.environ, CUDA_VISIBLE_DEVICES="0", PYTHONHASHSEED="0", CUBLAS_WORKSPACE_CONFIG=":4096:8",
           TOKENIZERS_PARALLELISM="false", PYTHONUNBUFFERED="1")
OUT_ROOT = Path(f"{REPO}/outputs/lgb16")
DRIVE_MOUNT = Path("/content/drive/MyDrive")
DRIVE_ROOT = DRIVE_MOUNT / "recoverkv_longgenbench16k"
CARD_DEFAULT = {"mu": 4, "v": 8, "t": 8, "vm": 4}       # the shipped gate card
LLAMA31_EOS = [128001, 128008, 128009]                  # generation_config eos: what vLLM stops on
JUDGE_PROMPT_SHA = hashlib.sha256(S.judge_prompt("<BLOCK>", "<DESC>").encode("utf-8")).hexdigest()[:16]
SMOKE_TYPE, SMOKE_NEW = "Menu Week", 2048               # its first document has checks on weeks 3 and 4
TOK_PER_DOC = 13000                                     # prior for the time projection
SEC_PER_TOKEN = {"fullkv": 0.030, "ours": 0.055, "ctrl_gate1.0": 0.060}   # priors; replaced by measurements
JUDGE_SEC_PER_CHECK = 1.3


class PartFailed(Exception):
    pass


# ----------------------------------------------------------------- settings and plan
def types():
    return list(TYPES) if TYPES else list(L.TYPES[SETTING])


def check_knobs():
    assert SETTING in L.SETTINGS, f"SETTING must be one of {L.SETTINGS}"
    for t in types():
        L.check_type(SETTING, t)
    assert int(N_PER_TYPE) >= 1 and int(CHUNK) >= 1
    assert DECODING in L.DECODINGS and JUDGE_DECODING in ("greedy", "repo_sampling")
    assert 0.0 <= QUANT_RATIO <= 1.0, "QUANT_RATIO in [0, 1]"
    assert 0.0 < GATE_RATIO <= 1.0, "GATE_RATIO in (0, 1]"
    assert 0.0 < CACHE_BUDGET <= 1.0, "CACHE_BUDGET in (0, 1]"
    assert QUANT_MODE in ("bytes", "tokens")
    assert DTYPE in ("float16", "bfloat16")
    if QUANT_RATIO > 0:
        assert WINDOW_SIZE % 4 == 0, "WINDOW_SIZE must be a multiple of 4 when QUANT_RATIO > 0"
    assert isinstance(LOCAL_WINDOW, int) and LOCAL_WINDOW % WINDOW_SIZE == 0, \
        "LOCAL_WINDOW must be an int multiple of WINDOW_SIZE"
    assert _git_head() == PIN, f"{REPO} is at {_git_head()[:10]}, not the pinned {PIN[:10]}: run step 2 again"
    L.dataset_sha(DATA_DIR, SETTING)                    # present and byte-identical to the pin


def arms():
    a = [("fullkv", None), ("ours", float(GATE_RATIO))]
    if CONTROL_ROW and QUANT_RATIO > 0 and GATE_RATIO < 1.0:
        a.append(("ctrl_gate1.0", 1.0))
    return a


def type_ids(t):
    return L.example_ids(DATA_DIR, SETTING, t, int(N_PER_TYPE))


def plan():
    parts = []
    for t in types():
        ids = type_ids(t)
        for c0 in range(0, len(ids), int(CHUNK)):
            c1 = min(c0 + int(CHUNK), len(ids))
            for arm, gate in arms():
                parts.append(dict(type=t, slug=L.type_slug(t), start=c0, end=c1, ids=ids[c0:c1],
                                  arm=arm, gate=gate))
    return parts


def op_tag():
    return (f"b{CACHE_BUDGET}_q{QUANT_RATIO}_w{WINDOW_SIZE}_s{NUM_SINK}_l{LOCAL_WINDOW}"
            f"_{QUANT_MODE}_{DTYPE}")


def part_dir(p, smoke=False):
    sub = f"fullkv_{DTYPE}" if p["arm"] == "fullkv" else f"{p['arm']}_g{p['gate']}_{op_tag()}"
    return OUT_ROOT / ("smoke_" + SETTING if smoke else SETTING) / sub / DECODING / \
        f"{p['slug']}_{p['start']:03d}-{p['end']:03d}"


def _git_head():
    r = subprocess.run(["git", "rev-parse", "HEAD"], cwd=REPO, capture_output=True, text=True)
    return r.stdout.strip() or "no_git"


def _gpu_name():
    try:
        r = subprocess.run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"],
                           capture_output=True, text=True)
    except OSError:
        return None
    return r.stdout.strip().splitlines()[0] if r.stdout.strip() else None


def part_settings(p, max_new=None):
    # Exactly what run_settings.json records; reuse compares against the same dict.
    s = dict(setting=SETTING, type=p["type"], start=p["start"], end=p["end"], ids=p["ids"],
             arm=p["arm"], model=MODEL_ID, dtype=DTYPE, decoding=DECODING,
             max_new_tokens_override=max_new, dataset_revision=L.HF_REVISION, commit=_git_head(),
             judge_model=JUDGE_MODEL, judge_decoding=JUDGE_DECODING, judge_prompt_sha=JUDGE_PROMPT_SHA)
    if p["arm"] != "fullkv":
        s.update(cache_budget=CACHE_BUDGET, window_size=WINDOW_SIZE, num_sink_tokens=NUM_SINK,
                 local_window_size=LOCAL_WINDOW, quant_ratio=QUANT_RATIO,
                 quant_budget_mode=QUANT_MODE, quant_gate_ratio=p["gate"], first_eviction_step=0)
    return s


def config_text(p, out, max_new=None):
    j = json.dumps
    lines = ["# generated by notebooks/longgenbench_colab.ipynb",
             "run: {mode: longgenbench, seed: 42}",
             f"model: {{name: {j(MODEL_ID)}, revision: null, dtype: {DTYPE}, "
             "attn_implementation: flash_attention_2}"]
    if p["arm"] == "fullkv":
        lines += ["cache: {backend: dynamic, backend_package: null, cache_budget: null}"]
    else:
        lines += ["cache:",
                  "  backend: windowed",
                  "  backend_package: flash_attn",
                  f"  cache_budget: {float(CACHE_BUDGET)}",
                  f"  window_size: {int(WINDOW_SIZE)}",
                  f"  num_sink_tokens: {int(NUM_SINK)}",
                  f"  local_window_size: {int(LOCAL_WINDOW)}",
                  f"  quant_ratio: {float(QUANT_RATIO)}",
                  f"  quant_budget_mode: {QUANT_MODE}",
                  f"  quant_gate_ratio: {float(p['gate'])}",
                  "  first_eviction_step: 0",
                  f"window: {{window_size: {int(WINDOW_SIZE)}, num_sink_tokens: {int(NUM_SINK)}, "
                  f"local_window_size: {int(LOCAL_WINDOW)}}}"]
    lines += ["telemetry: {track_scores: false}",
              "longgenbench:",
              f"  data_dir: {j(DATA_DIR)}",
              f"  setting: {j(SETTING)}",
              f"  types: [{j(p['type'])}]",
              f"  output_dir: {j(str(out))}",
              f"  start: {p['start']}",
              f"  end: {p['end']}",
              f"  decoding: {DECODING}",
              "  skip_oom: true",
              "  aggressive_cache_clear: true",
              f"  max_new_tokens: {'null' if max_new is None else int(max_new)}",
              ""]
    return "\n".join(lines)


# ----------------------------------------------------------------- files: always fresh
def _fresh(path):
    path = Path(path)
    if path.is_dir():
        shutil.rmtree(path)
    elif path.exists():
        path.unlink()
    return path


def _drive_path(local):
    return DRIVE_ROOT / Path(local).relative_to(OUT_ROOT)


def _to_drive(src):
    if not DRIVE_MOUNT.is_dir():
        return
    src = Path(src)
    dst = _fresh(_drive_path(src))
    dst.parent.mkdir(parents=True, exist_ok=True)
    shutil.copytree(src, dst) if src.is_dir() else shutil.copy2(src, dst)
    print("copied to", dst)


def _sha16(path):
    return hashlib.sha256(Path(path).read_bytes()).hexdigest()[:16]


# ----------------------------------------------------------------- the cache geometry, before any GPU hour
GEO_PY = r"""
import json, sys, statistics as st, torch
from transformers import AutoConfig, AutoTokenizer
from utils.cache_factory import get_cache_classes
from utils.prompting import encode_prompt
import data.longgenbench_loader as L
a = json.loads(sys.argv[1])
tok = AutoTokenizer.from_pretrained(a["model"]); mc = AutoConfig.from_pretrained(a["model"])
_, WCC, _ = get_cache_classes("flash_attn")
cc = WCC(window_size=a["w"], num_sink_tokens=a["s"], local_window_size=a["l"], cache_budget=a["b"],
         quant_ratio=a["q"], quant_budget_mode=a["mode"], quant_gate_ratio=a["g"], first_eviction_step=0)
mx = L.MAX_NEW_TOKENS[a["setting"]]
out = {}
for t in a["types"]:
    docs = L.load_type(a["data"], a["setting"], t, 0, a["n"])
    n = [len(encode_prompt(tok, d["prompt"], return_tensors="pt").input_ids[0]) for d in docs]
    rows = {}
    for name, P in (("min", min(n)), ("max", max(n))):
        r = cc.resolve(P, mc, torch.float16, mx)
        rows[name] = dict(prompt=P, budget_tokens=r.total_budget_tokens, N_q=r.N_q,
                          fp_cap=r.num_sink_tokens + r.local_tokens + r.top_k_fp * r.window_size,
                          cap=r.retained_tokens, util=round(r.budget_utilisation, 3))
    out[t] = rows
print("GEO" + json.dumps(out))
"""
_GEO = {}


def geometry():
    key = (SETTING, tuple(types()), int(N_PER_TYPE), op_tag(), GATE_RATIO)
    if key in _GEO:
        return _GEO[key]
    if QUANT_RATIO == 0:
        print("QUANT_RATIO = 0: no int2 tier, nothing to gate (an fp16-only ablation)")
    arg = json.dumps(dict(model=MODEL_ID, w=WINDOW_SIZE, s=NUM_SINK, l=LOCAL_WINDOW, b=CACHE_BUDGET,
                          q=QUANT_RATIO, mode=QUANT_MODE, g=GATE_RATIO, types=types(), data=DATA_DIR,
                          setting=SETTING, n=int(N_PER_TYPE)))
    r = subprocess.run([PY, "-c", GEO_PY, arg], cwd=REPO, env=ENV, capture_output=True, text=True)
    line = [l for l in r.stdout.splitlines() if l.startswith("GEO")]
    if r.returncode or not line:
        raise SystemExit("cache geometry preview failed:\n" + r.stderr[-3000:])
    geo = json.loads(line[0][3:])
    print(f"cache geometry at your settings ({SETTING}, max_new_tokens {L.MAX_NEW_TOKENS[SETTING]}):")
    print(f"  {'type':10s} {'prompt':>11s} {'budget tok':>11s} {'fp16 keys':>10s} {'int2 win':>9s} "
          f"{'keys kept':>10s} {'int2 from':>10s} {'drops from':>11s}")
    bad = []
    for t, rows in geo.items():
        for name in ("min", "max"):
            g = rows[name]
            print(f"  {t:10s} {name + ' ' + str(g['prompt']):>11s} {g['budget_tokens']:>11d} "
                  f"{g['fp_cap']:>10d} {g['N_q']:>9d} {g['cap']:>10d} "
                  f"{max(0, g['fp_cap'] - g['prompt']):>10d} {max(0, g['cap'] - g['prompt']):>11d}")
        if QUANT_RATIO > 0 and rows["min"]["N_q"] == 0:
            bad.append(f"{t}: no int2 window fits the budget (N_q = 0): no int2 tier, nothing to gate")
    print("  (fp16 keys = sink + local + fp windows. 'int2 from' / 'drops from' = generated tokens before the"
          "\n   int2 tier starts filling / before keys start being dropped.)")
    if bad:
        raise SystemExit("\n".join(bad))
    _GEO[key] = geo
    return geo


# ----------------------------------------------------------------- one part = generate (main.py) + judge
def _stream(cmd, log_path, keys):
    tail = []
    with open(log_path, "w") as log:
        proc = subprocess.Popen(cmd, cwd=REPO, env=ENV, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                                text=True, bufsize=1)
        for line in proc.stdout:
            log.write(line)
            tail = (tail + [line])[-40:]
            if any(k in line for k in keys):
                print("   ", line.rstrip()[:220])
        return proc.wait(), "".join(tail)


def run_part(p, out, max_new=None):
    out = Path(out)
    _fresh(out)
    if DRIVE_MOUNT.is_dir():
        _fresh(_drive_path(out))
    out.mkdir(parents=True)
    cfg = out / "_longgenbench.generated.yaml"
    cfg.write_text(config_text(p, out, max_new))
    (out / "run_settings.json").write_text(json.dumps(part_settings(p, max_new), indent=2))
    t0 = time.time()
    rc, tail = _stream([PY, "main.py", "--config", str(cfg)], out / "run.log",
                       ("new tokens in", "read gate", "compression:", "never filled", "THE READ GATE",
                        "Error", "error:", "Traceback", "WARNING"))
    (out / "wall_s.txt").write_text(f"{time.time() - t0:.1f}\n")
    if rc:
        print(tail)
        kind = "ran out of GPU memory" if "out of memory" in tail.lower() else f"main.py exited with {rc}"
        raise PartFailed(f"generation {kind}; full log: {out}/run.log")
    judge_part(p, out)
    return out


def judge_part(p, out):
    out = Path(out)
    t0 = time.time()
    rc, tail = _stream([PY, "-m", "modules.evaluation.longgenbench_judge", "--selftest",
                        "--model", JUDGE_MODEL, "--dtype", DTYPE, "--decoding", JUDGE_DECODING,
                        "--runs", str(out)], out / "judge.log",
                       ("selftest", "SELFTEST", "judged ", "Error", "Traceback"))
    (out / "judge_wall_s.txt").write_text(f"{time.time() - t0:.1f}\n")
    if rc:
        print(tail)
        raise PartFailed(f"judge exited with {rc} (a selftest miss exits 1); full log: {out}/judge.log")


# ----------------------------------------------------------------- checks
def _load(p, out):
    out = Path(out)
    meta = json.loads((out / f"{p['slug']}.meta.json").read_text())
    recs = S.read_jsonl(out / f"{p['slug']}.jsonl")
    return meta, recs


def _load_judge(p, out):
    out = Path(out)
    jm = json.loads((out / f"{p['slug']}.judge.meta.json").read_text())
    rows = S.read_jsonl(out / f"{p['slug']}.judge.jsonl")
    return jm, rows


def verify(p, out, max_new=None, quiet=False):
    # Every field a sidecar must carry for its number to belong to this part, arm and setting.
    fails, warns = [], []
    try:
        meta, recs = _load(p, out)
    except (OSError, ValueError) as e:
        return False, [f"generation incomplete: {e}"], []

    def eq(k, want, src=None, what=""):
        got = (meta if src is None else src).get(k)
        if got != want:
            fails.append(f"{what}{k} = {got!r}, expected {want!r}")

    eq("benchmark", "LongGenBench"); eq("setting", SETTING); eq("type", p["type"])
    eq("dataset_revision", L.HF_REVISION); eq("dataset_sha", L.FILES[SETTING][1])
    eq("prompt_format", L.PROMPT_FORMAT); eq("ids", p["ids"])
    eq("max_new_tokens", int(max_new or L.MAX_NEW_TOKENS[SETTING])); eq("max_new_tokens_override", max_new)
    eq("model_name", MODEL_ID); eq("dtype", DTYPE); eq("attn_implementation", "flash_attention_2")
    eq("transformers_version", "4.47.1")
    if meta.get("commit_sha") != PIN:
        fails.append(f"commit {str(meta.get('commit_sha'))[:10]} is not the pinned {PIN[:10]}")
    gen = meta.get("generation") or {}
    for k, v in (("decoding", DECODING), ("stop_string", L.STOP_STRING), ("stop_string_in_output", False),
                 ("eos_token_id", LLAMA31_EOS), ("output", "prefix + generation"), ("batch_size", 1)):
        eq(k, v, gen, "generation.")
    if [r.get("id") for r in recs] != p["ids"]:
        fails.append("records do not cover exactly this part's documents")
    n_null = sum(r.get("output") is None for r in recs)
    if n_null:
        warns.append(f"{n_null} document(s) ran out of GPU memory: excluded from every arm's paired numbers")
    rg = meta.get("read_gate") or {}
    if p["arm"] == "fullkv":
        eq("cache_type", "full_cache")
        if rg.get("verdict") != "not-expected":
            fails.append(f"full-cache read gate verdict {rg.get('verdict')!r}, expected 'not-expected'")
    else:
        eq("cache_type", "windowed"); eq("cache_backend_package", "flash_attn")
        for k, v in (("cache_budget", float(CACHE_BUDGET)), ("window_size", int(WINDOW_SIZE)),
                     ("num_sink_tokens", int(NUM_SINK)), ("local_window_size", int(LOCAL_WINDOW)),
                     ("quant_ratio", float(QUANT_RATIO)), ("quant_budget_mode", QUANT_MODE),
                     ("quant_gate_ratio", float(p["gate"])), ("first_eviction_step", 0),
                     ("quant_promotion", "bidir"), ("quant_promote_source", "dequant"),
                     ("quant_card_bits", CARD_DEFAULT)):
            eq(k, v)
        if (meta.get("quant_completion") or {}).get("mode") != "native":
            fails.append(f"quant_completion {meta.get('quant_completion')!r} is not the shipped kernel path")
        want_v = "gated" if QUANT_RATIO > 0 else "not-expected"
        if rg.get("verdict") != want_v:
            fails.append(f"read gate verdict {rg.get('verdict')!r}, expected {want_v!r} "
                         f"(gated={rg.get('gated')} fired={rg.get('fired')})")
        if want_v == "gated":
            if not rg.get("fired") or rg.get("gated") != rg.get("fired"):
                fails.append(f"gate ran on {rg.get('gated')} of {rg.get('fired')} fused layers")
            if rg.get("completion_applied"):
                fails.append("the completion-ablation epilogue ran: not the shipped method")
            rf = rg.get("read_fraction")
            if rf is None or abs(rf - float(p["gate"])) > READ_FRACTION_TOL:
                warns.append(f"realised read fraction {rf} vs configured {p['gate']}")
        geo = meta.get("resolved_geometry_first_example") or {}
        if QUANT_RATIO > 0 and not geo.get("N_q"):
            fails.append(f"NO int2 tier at this shape (N_q = {geo.get('N_q')}): nothing to gate")
        for k, v in (("window_size", int(WINDOW_SIZE)), ("num_sink_tokens", int(NUM_SINK)),
                     ("local_tokens", int(LOCAL_WINDOW)), ("quant_budget_mode", QUANT_MODE)):
            eq(k, v, geo, "geometry.")
        if meta.get("n_examples_no_int2"):
            warns.append(f"{meta['n_examples_no_int2']} document(s) never filled past the fp16 tier "
                         "(no int2 window, no gate on those)")
    # The judge: the right judge, on exactly these generations, every eligible check asked.
    try:
        jm, rows = _load_judge(p, out)
    except (OSError, ValueError) as e:
        fails.append(f"judge incomplete: {e}")
    else:
        for k, v in (("judge_model", JUDGE_MODEL), ("judge_decoding", JUDGE_DECODING),
                     ("judge_max_new_tokens", S.JUDGE_MAX_NEW_TOKENS), ("judge_batch_size", 1),
                     ("judge_prompt_sha", JUDGE_PROMPT_SHA), ("judge_dtype", DTYPE),
                     ("ids", p["ids"]), ("transformers_version", "4.47.1")):
            eq(k, v, jm, "judge.")
        if jm.get("commit_sha") != PIN:
            fails.append(f"judge ran at commit {str(jm.get('commit_sha'))[:10]}, not {PIN[:10]}")
        if jm.get("generations_sha") != _sha16(Path(out) / f"{p['slug']}.jsonl"):
            fails.append("the judge's verdicts are for a different generations file")
        want_n = sum(1 for r in recs if r.get("output") is not None
                     for c in S.document_checks(r) if c["block"] is not None)
        if len(rows) != want_n:
            fails.append(f"judge answered {len(rows)} checks, the documents have {want_n} eligible")
        if rows and jm.get("n_strict_disagree", 0) > 0.2 * len(rows):
            warns.append(f"judge: the first word disagrees with the 'yes'-in-response rule on "
                         f"{jm['n_strict_disagree']}/{len(rows)} checks; read judge.jsonl")
    if not quiet:
        for w in warns:
            print("  warn:", w)
        for f in fails:
            print("  FAIL:", f)
    return not fails, fails, warns


def report_part(p, out, max_new=None):
    meta, recs = _load(p, out)
    try:
        _, rows = _load_judge(p, out)
    except (OSError, ValueError):
        rows = []
    s = S.summarize(recs, S.judgments_by_doc(rows))
    rg = meta.get("read_gate") or {}
    stops = {k: sum(r.get("stopped_on") == k for r in recs) for k in ("stop_string", "eos", "max_new_tokens")}
    line = (f"  {p['type']} [{p['start']}, {p['end']}) {p['arm']}: CR {s['CR']:.1f}  STIC-1 {s['STIC-1']:.1f}  "
            f"STIC-2 {s['STIC-2']:.1f}  wAvg {s['wAvg']:.1f}  ({s['n_yes']}/{s['n_judged']} judged yes of "
            f"{s['n_checks']})\n    new tokens {meta.get('total_gen_tokens')}  stopped on {stops}  "
            f"peak {meta.get('peak_mem_gib_max')} GiB")
    if p["arm"] != "fullkv":
        rf = rg.get("read_fraction")
        line += (f"\n    gate {rg.get('verdict')} ({rg.get('gated')}/{rg.get('fired')} layers, read "
                 f"{'n/a' if rf is None else f'{rf:.3f}'})  int2 unused on {meta.get('n_examples_no_int2')}/"
                 f"{len(recs)}  no key dropped on {meta.get('n_examples_no_eviction')}/{len(recs)}  KV bytes <= "
                 f"{meta.get('mean_kv_bytes_fraction_upper')} of full")
    wall = Path(out) / "wall_s.txt"
    if wall.is_file():
        line += f"\n    generation {float(wall.read_text()) / 60:.1f} min"
    print(line)
    ok, _, _ = verify(p, out, max_new)
    print("  VERIFIED" if ok else "  NOT VERIFIED: do not quote this part")
    return ok


def _restore(p, out):
    # A part that is on Drive but not on this disk (new runtime): bring it back.
    out = Path(out)
    if (not (out / f"{p['slug']}.meta.json").is_file() and DRIVE_MOUNT.is_dir()
            and _drive_path(out).is_dir()):
        shutil.copytree(_drive_path(out), out, dirs_exist_ok=True)
        print("  found in Drive:", out.relative_to(OUT_ROOT))
    return out


def _reusable(p, out):
    if not REUSE_VERIFIED:
        return False
    out = _restore(p, out)
    try:
        st = json.loads((out / "run_settings.json").read_text())
    except (OSError, ValueError):
        return False
    want = part_settings(p)
    diff = {k: (st.get(k), v) for k, v in want.items() if st.get(k) != v}
    judge_keys = {"judge_model", "judge_decoding", "judge_prompt_sha"}
    if diff and set(diff) <= judge_keys and (out / f"{p['slug']}.meta.json").is_file():
        # Only the judge changed: keep the generations, judge again.
        print("  generations reusable, judge settings changed: judging again", diff)
        st.update({k: want[k] for k in judge_keys})
        (out / "run_settings.json").write_text(json.dumps(st, indent=2))
        judge_part(p, out)
        diff = {}
    if diff:
        print("  earlier result not reusable, settings differ (was, now):", diff)
        return False
    try:
        meta, _ = _load(p, out)
    except (OSError, ValueError):
        print("  earlier result not reusable: incomplete")
        return False
    if meta.get("gpu_name") != _gpu_name():
        print(f"  earlier result not reusable: recorded on {meta.get('gpu_name')}, this GPU is {_gpu_name()}")
        return False
    ok, fails, _ = verify(p, out, quiet=True)
    if not ok and all(f.startswith("judge") or "judge" in f for f in fails):
        print("  generations verify, the judge does not: judging again")
        judge_part(p, out)
        ok, fails, _ = verify(p, out, quiet=True)
    return ok


# ----------------------------------------------------------------- time projection
def _projection(remaining, measured, tok_per_doc):
    sec = 0.0
    for p in remaining:
        spt = measured.get(p["arm"]) or SEC_PER_TOKEN[p["arm"]]
        n = len(p["ids"])
        sec += n * tok_per_doc * spt + n * 15 * JUDGE_SEC_PER_CHECK + 150.0   # + two model loads
    return sec / 3600


# ----------------------------------------------------------------- the whole run
def run_all():
    check_knobs()
    geometry()
    parts = plan()
    print(f"\nGPU: {_gpu_name()} | commit: {_git_head()[:8]} | Drive: "
          f"{'mounted' if DRIVE_MOUNT.is_dir() else 'NOT mounted (results stay local)'}")
    print(f"{len(parts)} parts: {len(types())} types x {int(N_PER_TYPE)} documents x {len(arms())} arms; "
          f"projected {_projection(parts, {}, TOK_PER_DOC):.1f} h if nothing is reused")
    failed, gen_tok, gen_sec, docs_done = [], {}, {}, []
    measured = {}
    for k, p in enumerate(parts):
        out = part_dir(p)
        print(f"\n#################### part {k + 1}/{len(parts)}: {SETTING} {p['type']} "
              f"[{p['start']}, {p['end']}) {p['arm']} ####################")
        if _reusable(p, out):
            print("  reusing the verified earlier result, not re-run")
            continue
        try:
            run_part(p, out)
        except PartFailed as e:
            failed.append(f"{p['type']} [{p['start']}, {p['end']}) {p['arm']}: {e}")
            print(f"\n!! {failed[-1]}\n   flagged; continuing with the next part")
            _to_drive(out)
            continue
        if not report_part(p, out):
            failed.append(f"{p['type']} [{p['start']}, {p['end']}) {p['arm']}: did not verify")
        _to_drive(out)
        meta, recs = _load(p, out)
        gen_tok[p["arm"]] = gen_tok.get(p["arm"], 0) + int(meta.get("total_gen_tokens") or 0)
        gen_sec[p["arm"]] = gen_sec.get(p["arm"], 0.0) + float((out / "wall_s.txt").read_text())
        docs_done += [r.get("gen_tokens") or 0 for r in recs]
        measured = {a: gen_sec[a] / gen_tok[a] for a in gen_tok if gen_tok[a]}
        tpd = float(np.mean(docs_done)) if docs_done else TOK_PER_DOC
        print("  measured: " + ", ".join(f"{a} {1e3 * v:.0f} ms/token (part wall clock)" for a, v in measured.items())
              + f", {tpd:.0f} new tokens/document | remaining ~{_projection(parts[k + 1:], measured, tpd):.1f} h")
    return final(failed)


# ----------------------------------------------------------------- the table
def _collect(arm):
    # {id: record}, {id: [verdict rows]}, [metas], [unverified] for one arm, over the plan.
    recs, judg, metas, bad = {}, {}, [], []
    for p in plan():
        if p["arm"] != arm:
            continue
        out = _restore(p, part_dir(p))
        ok, fails, _ = verify(p, out, quiet=True) if out.is_dir() else (False, ["not run yet"], [])
        if not ok:
            bad.append(f"{p['type']} [{p['start']}, {p['end']}) {arm}: {fails[0] if fails else '?'}")
            continue
        meta, rs = _load(p, out)
        _, rows = _load_judge(p, out)
        metas.append(meta)
        recs.update({r["id"]: r for r in rs})
        for i, v in S.judgments_by_doc(rows).items():
            judg[i] = v
    return recs, judg, metas, bad


def _fmt(x, n=1):
    return "-" if x is None or (isinstance(x, float) and x != x) else f"{x:.{n}f}"


def final(failed=()):
    check_knobs()
    data = {arm: _collect(arm) for arm, _ in arms()}
    unverified = [u for arm in data for u in data[arm][3]]
    notes = list(failed)
    summary = {"setting": SETTING, "decoding": DECODING, "operating_point": op_tag(), "gate_ratio": GATE_RATIO,
               "judge": JUDGE_MODEL, "judge_decoding": JUDGE_DECODING, "commit": _git_head(), "gpu": _gpu_name(),
               "rows": {}}
    hdr = (f"{'scope':10s} {'arm':13s} {'docs':>4s} {'CR':>6s} {'STIC-1':>7s} {'STIC-2':>7s} {'wAvg':>6s} "
           f"{'once':>6s} {'range':>6s} {'period':>6s} {'Len':>6s} {'S2strict':>8s} {'newtok':>7s} {'cap%':>5s} "
           f"{'KVbytes<=':>9s} {'read':>6s} {'tok/s*':>7s}")
    lines = [hdr, "-" * len(hdr)]
    deltas = []
    for scope in types() + ["ALL"]:
        tl = types() if scope == "ALL" else [scope]
        ids = [i for t in tl for i in type_ids(t)
               if all(i in data[a][0] and data[a][0][i].get("output") is not None for a, _ in arms())]
        if not ids:
            lines.append(f"{scope:10s} (no document has a verified result in every arm yet)")
            continue
        per_arm = {}
        for arm, _ in arms():
            recs, judg, metas, _ = data[arm]
            docs = [recs[i] for i in ids]
            s = S.summarize(docs, {i: judg.get(i, []) for i in ids})
            per_arm[arm] = (docs, {i: judg.get(i, []) for i in ids})
            kvb = [d.get("kv_bytes_fraction_upper") for d in docs if d.get("kv_bytes_fraction_upper") is not None]
            ms = [m for m in metas if m.get("type") in tl]
            rd = [m["read_gate"]["read_fraction"] for m in ms if (m.get("read_gate") or {}).get("read_fraction")]
            tg = sum(d.get("gen_tokens") or 0 for d in docs)
            tw = sum(d.get("wall_s") or 0 for d in docs)
            cap = 100.0 * np.mean([bool(d.get("hit_token_cap")) for d in docs])
            lines.append(f"{scope:10s} {arm:13s} {len(ids):>4d} {_fmt(s['CR']):>6s} {_fmt(s['STIC-1']):>7s} "
                         f"{_fmt(s['STIC-2']):>7s} {_fmt(s['wAvg']):>6s} {_fmt(s['acc_once']):>6s} "
                         f"{_fmt(s['acc_range']):>6s} {_fmt(s['acc_periodic']):>6s} "
                         f"{(str(s['Len']) if s['Len'] else '-'):>6s} {_fmt(s['STIC-2_strict']):>8s} "
                         f"{tg / len(ids):>7.0f} {cap:>5.0f} {(np.mean(kvb) if kvb else 1.0):>9.3f} "
                         f"{(f'{np.mean(rd):.3f}' if rd else '-'):>6s} {(f'{tg / tw:.1f}' if tw else '-'):>7s}")
            summary["rows"][f"{scope}/{arm}"] = {**s, "n_docs": len(ids),
                                                 "kv_bytes_fraction_upper": float(np.mean(kvb)) if kvb else 1.0}
        for arm, _ in arms():
            if arm == "fullkv":
                continue
            d = S.bootstrap_delta(*per_arm["fullkv"], *per_arm[arm])
            deltas.append(f"{scope:10s} {arm} - fullkv: " + "   ".join(
                f"{k} {v['delta']:+.1f} [{v['lo']:+.1f}, {v['hi']:+.1f}]" for k, v in d.items()))
            summary["rows"][f"{scope}/{arm}"]["delta_vs_fullkv"] = d
            if DECODING == "greedy":
                fk, ou = per_arm["fullkv"][0], per_arm[arm][0]
                pref = [len(os.path.commonprefix([a["output"], b["output"]])) for a, b in zip(fk, ou)]
                same = sum(a["output"] == b["output"] for a, b in zip(fk, ou))
                deltas.append(f"{'':10s}   text identical to fullkv for a median {int(np.median(pref))} characters "
                              f"(min {min(pref)}); {same}/{len(ids)} documents identical end to end")
        lines.append("")
    pub = L.PUBLISHED_LLAMA31_8B[SETTING]
    lines.append(f"{'paper':10s} {'Llama3.1-8B':13s} {400:>4d} {pub['CR']:>6.1f} {pub['STIC-1']:>7.1f} "
                 f"{pub['STIC-2']:>7.1f} {pub['wAvg']:>6.1f} {pub['acc_once']:>6.1f} {pub['acc_range']:>6.1f} "
                 f"{pub['acc_periodic']:>6.1f} {pub['Len']:>6d}   (Table 2: vLLM, bf16, all 400; context only)")
    lines.append("\nours - fullkv on the same documents, [95% CI] from a paired bootstrap over documents:")
    lines += ["  " + x for x in deltas]
    lines.append("\nCR, STIC-1, STIC-2, wAvg, once/range/period in %. STIC-1 = yes / judged, STIC-2 = yes / all "
                 "instructions,\nwAvg = CR x STIC-2. S2strict = STIC-2 with the judge's first word as the verdict (a "
                 "diagnostic).\nLen = mean words over outputs with > 40 entries and > 2000 words. cap% = documents that "
                 "hit max_new_tokens.\n* tok/s is wall clock inside a quality run (prefill, compiles and Python "
                 "included): context, not a performance number.\n  KVbytes<= is an upper bound on the KV bytes held "
                 "at the end of each document, over the full cache's, averaged.")
    if notes:
        lines.append("\nflagged this session:")
        lines += ["  " + n for n in notes]
    if unverified:
        lines.append("\nNOT in the table (missing or did not verify; rerun step 11):")
        lines += ["  " + u for u in unverified]
    table = "\n".join(lines)
    path = _fresh(OUT_ROOT / f"{SETTING}_FINAL_{DECODING}_{op_tag()}_g{GATE_RATIO}_n{int(N_PER_TYPE)}.txt")
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(table + "\n")
    path.with_suffix(".json").write_text(json.dumps(summary, indent=2, default=str) + "\n")
    print("\n=================== FINAL TABLE (paired documents) ===================")
    print(table)
    ok = not unverified and not failed
    print("\nEVERY PART RAN AND VERIFIED." if ok else
          "\nNOT EVERY PART VERIFIED. Run step 11 again: verified parts are reused, only the others re-run.")
    print("written to", path)
    _to_drive(path)
    _to_drive(path.with_suffix(".json"))
    return ok


check_knobs()
print("helpers loaded | parts:", len(plan()), "| GPU:", _gpu_name(), "| commit:", _git_head()[:8])

## 10. Smoke run (~15 min). Must print `SMOKE OK`
Before any GPU hour, this prints the cache geometry at your settings for the planned prompts, and refuses a setting
with no int2 tier.

Then it runs one Menu Week document per arm, capped at 2,048 new tokens, through the same code path as step 11. That
document has instructions on weeks 3 and 4, so the judge gets real entries to score. This checks the plumbing, not the
score:
* every sidecar field;
* the gate verdict, and that the int2 tier was used and keys were dropped;
* the judge's selftest and its file;
* how far ours and fullkv agree character for character.

In [ ]:
check_knobs()
geometry()
first = type_ids(SMOKE_TYPE)[:1]
ok_all, smoke_out = True, {}
for arm, gate in arms():
    p = dict(type=SMOKE_TYPE, slug=L.type_slug(SMOKE_TYPE), start=0, end=1, ids=first, arm=arm, gate=gate)
    out = part_dir(p, smoke=True)
    print(f"\n---- smoke: {arm} ----")
    run_part(p, out, max_new=SMOKE_NEW)
    ok_all &= report_part(p, out, max_new=SMOKE_NEW)
    meta, recs = _load(p, out)
    smoke_out[arm] = recs[0]
    if arm != "fullkv":
        r = recs[0]
        print(f"  int2 tier used: {r.get('int2_used')}   keys dropped: {r.get('evicted')}   "
              f"(fp16 holds {r.get('fp_tokens_cap')} keys, both tiers {r.get('retained_tokens_cap')})")
        ok_all &= bool(r.get("int2_used")) and bool(r.get("evicted"))
if DECODING == "greedy":
    a, b = smoke_out["fullkv"]["output"], smoke_out["ours"]["output"]
    n = len(os.path.commonprefix([a, b]))
    print(f"\nours vs fullkv on the same prompt: identical for the first {n} characters "
          f"(of {len(a)} / {len(b)}); greedy, so they agree until the compressed cache changes an argmax")
assert ok_all, "smoke run did not verify: do not start step 11"
print("\nSMOKE OK")

## 11. The real run
For each type and each chunk of documents, fullkv runs first, then ours (then the control, if on). Each part is
generated, judged (a separate process with a clean full-cache model, after its selftest), printed, verified and copied
to Drive before the next one starts. The remaining time is re-projected from measured speed after each part. A part
that fails a check or runs out of GPU memory is flagged, and the next part runs. Run this cell again at any point:
parts that already verified are reused, and only the others run. Changing only the judge settings re-judges the saved
generations without regenerating them.

Expected and harmless in the output: `operation scheduled before its operands`, `tl.where with a non-boolean
condition`, the TF32 note, `failed during evaluate_expr(u0 <= 0 ...)`, and transformers' notes that
`temperature`/`top_p` are ignored when `do_sample=False`.

In [ ]:
OK = run_all()

## 12. Reprint the final table
After a session restart, run steps 7–9 first. On a brand-new runtime, run steps 1–5 and 7–9: the parts come back from
Drive, and nothing is re-run. Only documents with a verified, judged result in every arm enter the table.

In [ ]:
final()